# Crawling PTA

*Web crawler untuk mengambil semua data skripsi di PTA Trunojoyo (khusus prodi teknik).*

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sys, time

def ptaa():
    data = {
        "penulis": [],
        "judul": [],
        "pembimbing_pertama": [],
        "pembimbing_kedua": [],
        "abstrak_bindonesia": [],
        "abstrak_binggris": [],
        "prodi_id": [],
        "nama_prodi": []
    }

    # daftar prodi Fakultas Teknik
    prodi_teknik = [9, 10, 11, 19, 20, 23, 31, 32, 33]

    for prodi_id in prodi_teknik:
        page = 1
        nama_prodi = ""
        total_pages = 0
        start_time = time.time()

        while True:
            url = f"https://pta.trunojoyo.ac.id/c_search/byprod/{prodi_id}/{page}"
            r = requests.get(url)
            soup = BeautifulSoup(r.content, "html.parser")
            jurnals = soup.select('li[data-cat="#luxury"]')

            # kalau tidak ada data → berhenti
            if not jurnals:
                break

            if not nama_prodi:
                prodi_header = soup.select_one("h1") or soup.select_one("div.title") or soup.select_one("div#begin h2")
                nama_prodi = prodi_header.get_text(strip=True) if prodi_header else f"Prodi {prodi_id}"

            for jurnal in jurnals:
                detail_url = jurnal.select_one('a.gray.button')['href']
                response = requests.get(detail_url)
                soup1 = BeautifulSoup(response.content, "html.parser")
                isi = soup1.select_one('div#content_journal')

                judul = isi.select_one('a.title').text.strip()

                # ambil penulis & pembimbing
                def ambil_span(teks):
                    el = isi.find("span", string=lambda t: t and teks in t)
                    return el.get_text(" ", strip=True).split(":", 1)[-1].strip() if el else ""

                penulis = ambil_span("Penulis")
                pembimbing_pertama = ambil_span("Dosen Pembimbing I")
                pembimbing_kedua = ambil_span("Dosen Pembimbing II")

                abstrak_paragraf = isi.find_all('p', align="justify")
                abstrak_bindonesia = abstrak_paragraf[0].get_text(strip=True) if len(abstrak_paragraf) > 0 else ""
                abstrak_binggris = abstrak_paragraf[1].get_text(strip=True) if len(abstrak_paragraf) > 1 else ""

                # masukkan data
                data["penulis"].append(penulis)
                data["judul"].append(judul)
                data["pembimbing_pertama"].append(pembimbing_pertama)
                data["pembimbing_kedua"].append(pembimbing_kedua)
                data["abstrak_bindonesia"].append(abstrak_bindonesia)
                data["abstrak_binggris"].append(abstrak_binggris)
                data["prodi_id"].append(prodi_id)
                data["nama_prodi"].append(nama_prodi)

            total_pages = page
            # tampilkan progress di satu baris
            sys.stdout.write(f"\r[{prodi_id}] {nama_prodi} - Sedang ambil halaman {page} ...")
            sys.stdout.flush()
            page += 1

        # selesai 1 prodi
        sys.stdout.write(f"\r [{prodi_id}] {nama_prodi} selesai! Total halaman: {total_pages}\n")
        sys.stdout.flush()

    df = pd.DataFrame(data)
    df.to_csv("PPW_HasilCrawling_FakultasTeknik.csv", index=False, encoding="utf-8-sig")

    print(f"\n Total entri keseluruhan: {len(df)}")
    return df

In [ ]:
ptaa()

[9] Teknik Industri - Page 1/1 [] 100.00%

[9] Teknik Industri - Page 2/2 [] 100.00%

[9] Teknik Industri - Page 3/3 [] 100.00%

[9] Teknik Industri - Page 4/4 [] 100.00%

[9] Teknik Industri - Page 5/5 [] 100.00%

[9] Teknik Industri - Page 6/6 [] 100.00%

[9] Teknik Industri - Page 7/7 [] 100.00%

[9] Teknik Industri - Page 8/8 [] 100.00%

[9] Teknik Industri - Page 9/9 [] 100.00%

[9] Teknik Industri - Page 10/10 [] 100.00%

[9] Teknik Industri - Page 11/11 [] 100.00%

[9] Teknik Industri - Page 12/12 [] 100.00%

[9] Teknik Industri - Page 13/13 [] 100.00%

[9] Teknik Industri - Page 14/14 [] 100.00%

[9] Teknik Industri - Page 15/15 [] 100.00%

[9] Teknik Industri - Page 16/16 [] 100.00%

[9] Teknik Industri - Page 17/17 [] 100.00%

[9] Teknik Industri - Page 18/18 [] 100.00%

[9] Teknik Industri - Page 19/19 [] 100.00%

[9] Teknik Industri - Page 20/20 [] 100.00%

[9] Teknik Industri - Page 21/21 [] 100.00%

[9] Teknik Industri - Page 22/22 [] 100.00%

[9] Teknik Industri - Page 2

*Melanjutkan proses yang sempat berhenti di Prodi tertentu atau halaman tertentu.*

In [5]:
import requests, time, sys
from bs4 import BeautifulSoup
import pandas as pd

def ptaa(start_index=0, start_page=1):
    data = {
        "penulis": [], "judul": [], "pembimbing_pertama": [], "pembimbing_kedua": [],
        "abstrak_bindonesia": [], "abstrak_binggris": [], "prodi_id": [], "nama_prodi": []
    }

    prodi_teknik = [9,10,11,19,20,23,31,32,33]

    for idx, prodi_id in enumerate(prodi_teknik[start_index:], start=start_index):
        page = start_page if idx==start_index else 1   # <– pakai start_page cuma di prodi pertama
        nama_prodi = ""
        total_pages = 0

        while True:
            url = f"https://pta.trunojoyo.ac.id/c_search/byprod/{prodi_id}/{page}"
            r = requests.get(url, timeout=30)
            soup = BeautifulSoup(r.content, "html.parser")
            jurnals = soup.select('li[data-cat="#luxury"]')

            if not jurnals:
                break

            if not nama_prodi:
                prodi_header = soup.select_one("h1") or soup.select_one("div.title") or soup.select_one("div#begin h2")
                nama_prodi = prodi_header.get_text(strip=True) if prodi_header else f"Prodi {prodi_id}"

            for jurnal in jurnals:
                detail_url = jurnal.select_one('a.gray.button')['href']
                response = requests.get(detail_url, timeout=30)
                soup1 = BeautifulSoup(response.content, "html.parser")
                isi = soup1.select_one('div#content_journal')

                judul = isi.select_one('a.title').text.strip()

                def ambil_span(teks):
                    el = isi.find("span", string=lambda t: t and teks in t)
                    return el.get_text(" ", strip=True).split(":", 1)[-1].strip() if el else ""

                penulis = ambil_span("Penulis")
                pembimbing_pertama = ambil_span("Dosen Pembimbing I")
                pembimbing_kedua = ambil_span("Dosen Pembimbing II")

                abstrak_paragraf = isi.find_all('p', align="justify")
                abstrak_bindonesia = abstrak_paragraf[0].get_text(strip=True) if len(abstrak_paragraf) > 0 else ""
                abstrak_binggris = abstrak_paragraf[1].get_text(strip=True) if len(abstrak_paragraf) > 1 else ""

                data["penulis"].append(penulis)
                data["judul"].append(judul)
                data["pembimbing_pertama"].append(pembimbing_pertama)
                data["pembimbing_kedua"].append(pembimbing_kedua)
                data["abstrak_bindonesia"].append(abstrak_bindonesia)
                data["abstrak_binggris"].append(abstrak_binggris)
                data["prodi_id"].append(prodi_id)
                data["nama_prodi"].append(nama_prodi)

            total_pages = page
            sys.stdout.write(f"\r[{prodi_id}] {nama_prodi} - Page {page}")
            sys.stdout.flush()
            page += 1

        sys.stdout.write(f"\r[{prodi_id}] {nama_prodi} selesai! Total halaman: {total_pages}\n")
        sys.stdout.flush()

        df_partial = pd.DataFrame(data)
        df_partial.to_csv("PPW_HasilCrawling_FakultasTeknik_partial.csv", index=False, encoding="utf-8-sig")

    df = pd.DataFrame(data)
    df.to_csv("PPW_HasilCrawling_FakultasTeknik.csv", index=False, encoding="utf-8-sig")
    print(f"\nTotal entri keseluruhan: {len(df)}")
    return df

In [6]:
ptaa(start_index=4, start_page=19)

[20] Journal Jurusan Mekatronika selesai! Total halaman: 28
[23] Journal Jurusan Teknik Elektro selesai! Total halaman: 34
[31]  selesai! Total halaman: 0
[32]  selesai! Total halaman: 0
[33] Journal Jurusan Teknik Mekatronika selesai! Total halaman: 3

Total entri keseluruhan: 228


,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak_bindonesia,abstrak_binggris,prodi_id,nama_prodi
0,R. Agung Wicaksono,Sistem Monitoring dan Manajemen Baterai Mobil ...,"Ahmad Sahru Romadhon, S.kom, M.T.",,"Saat ini, pengembangan mobil listrik di Indone...","At present, the development of electric cars i...",20,Journal Jurusan Mekatronika
1,M. Wildan Firdausy,Prototipe Lori Pengangkut Tebu Otomatis,"Sri Wahyuni, S.Kom, M.T.",,Pada wilayah Indonesia pada musim panen tebu s...,In the area of Indonesia during the sugarcane ...,20,Journal Jurusan Mekatronika
2,Faisal Edy Fitrullah,Penentuan Pergerakan Robotsoccer Terhadap Bola...,"Faikul Umam, S.Kom., M.T",,"Perkembangan teknologi didunia begitu pesat, t...",The development of technology in the world is ...,20,Journal Jurusan Mekatronika
3,Chairul Anam,ALAT PENCUCI DAN PENGERING TANGAN OTOMATIS,"Hairil Budiarto., S.T., M.T",,"Mencuci tangan merupakan hal sederhana, namun ...","Hand washing is a simple thing, but has a very...",20,Journal Jurusan Mekatronika
4,Septian Pamu Wardana,Perancangan Sistem Jarak Pada Mobil Listrik,"Ahmad Sahru Romadhon., S.kom., M.T.",,Mobil listrik semakin banyak digunakan untuk m...,Electric cars are increasingly being used to r...,20,Journal Jurusan Mekatronika
...,...,...,...,...,...,...,...,...
223,MOH TAUFIK HIDAYAT,ALAT PENGGULUNG UNTUK MENGATUR KERAPATAN KERTA...,"FAIKUL UMAM., S.KOM., M.T","SRI WAHYUNI., S.KOM., M.T",ABSTRAK\n\nKertas merupakan kebutuhan yang dib...,ABSTRACT\n\nPaper is a necessity needed by hum...,33,Journal Jurusan Teknik Mekatronika
224,Alvian Ainun Fajrih,Optimasi Penguapan Air Laut Tua Pada Rumah Kaca,"Hairil Budiarto, S.T., M.T.","Faikul Umam, S.Kom., M.T.",Pembuatan garam di Madura umumnya dilakukan de...,Making salt in Madura is generally done by hea...,33,Journal Jurusan Teknik Mekatronika
225,ADINDA DEBTIANA DWIKA HILDA,KESTABILAN KECEPATAN MOBILE ROBOT PADA LINTASA...,"Faikul Umam, S.Kom., M.T.","Sri Wahyuni, S.Kom., M.T.",Mobile robot merupakan salah satu kategori rob...,Mobile robot is one of the robot categories th...,33,Journal Jurusan Teknik Mekatronika
226,Nafizatul Jamilah,Rancang Bangun Sistem Otomasi Robot Pengecatan...,"Hairil Budiarto, S.T., M.T","Sri Wahyuni, S.Kom., M.T",ABSTRAK\n\nMarka jalan raya adalah sebuah tand...,ABSTRACT\n\nA highway mark is a sign that can ...,33,Journal Jurusan Teknik Mekatronika
